# Guided exercise: Meera's question in one statement

**Week 0 foundations guide, Chapter 2, SQL. Hands-on, 90 minutes.** This notebook builds mini
project 2 one query at a time: four queries against a small orders table, each checked as it runs,
then the chapter's three traps staged on the same rows.

**The setup this notebook needs.** Postgres running, with the practice tables loaded into a
database of their own called `foundations`. In a new Codespace, work through steps 1, 2, 3 and 6 of
the [Postgres setup sheet](../../../D4/whiteboards/C2_W00_D04_postgres_setup_STUDENT.md) from Week 0,
Thursday; if `psql -d postgres -c "SELECT 1"` already answers in your terminal, Postgres is running
and those steps are done. Then run these two commands from the folder that holds the
[setup file](../../sql/C2_W00_D02_foundations_02_orders_setup_STUDENT.sql), which in this
repository is `content/W00/D2/sql/`:

```bash
createdb foundations
psql -d foundations -f C2_W00_D02_foundations_02_orders_setup_STUDENT.sql
```

The second command ends by printing 5, 12, 16, 1 and 2. Running it again rebuilds the tables from
scratch, so a mistake is never permanent.

**How the TODOs work.** Each TODO is a lettered choice written in the comment above a `__TODOn__`
placeholder: replace the placeholder with the code of the letter you pick, run the cell, and the
checks under it say whether you were right. A PREDICT TODO takes your prediction before the query
runs, and the trace in Step 4 takes numbers you work out by hand. Run as shipped, the notebook stops
at the first placeholder with a `NameError`, which is where it is meant to start.

The next cell finds the programme's helper, `scripts/c2kit.py`, by walking up from this folder,
points it at the `foundations` database with the same connection settings `psql` uses, and draws
where this exercise sits among the guide's eight.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

# The practice tables live in their own database, foundations.
kit.WAREHOUSE.clear()
kit.WAREHOUSE.update(dbname="foundations")

kit.side_by_side(
    kit.ladder(["Python", "SQL", "Numbers", "Language models", "Business problems", "pandas",
                "Excel", "GitHub"], lit=1, title="The guide's eight hands-on exercises", show=False),
    kit.vflow(["1. Meera's question", "2. Redraw the pipeline", "3. Predict, then run four queries",
               "4. Trace one query", "5. Break it on purpose", "6. The mini project's self-check"],
              title="This exercise, in six steps", show=False),
)

## Step 1. Meera's question, and the tables (5 minutes)

Meera Raghavan, CEO of Kalpa Retail, asked the question the whole guide follows:

> "Revenue fell from Q1 to Q2, where did it go?"

Chapter 2 ends on writing the query that answers her in one statement. The tables here are a
practice miniature of Kalpa's orders, written by hand for this exercise and not Kalpa's data: twelve
orders across the Plus, Basic and Student tiers in the first two quarters of 2026, one of them with
no amount and two of them cancelled, and five customers, one of whom never ordered. Orders 7 to 11
are the five rows of the diagnostic's Q13 table. A third table, `order_items`, holds the item lines
on each order and waits for Step 5.

In [ ]:
kit.sql_table("SELECT * FROM orders ORDER BY order_id", caption="orders: twelve rows of practice data")
kit.sql_table("SELECT * FROM customers ORDER BY customer_id", caption="customers: five rows")

In [ ]:
loaded = kit.sql("""
    SELECT (SELECT COUNT(*) FROM orders)                            AS orders,
           (SELECT COUNT(*) FROM customers)                         AS customers,
           (SELECT COUNT(*) FROM order_items)                       AS item_lines,
           (SELECT COUNT(*) FROM orders WHERE amount IS NULL)       AS missing_amounts,
           (SELECT COUNT(*) FROM orders WHERE status = 'cancelled') AS cancelled
""")[0]
print(loaded)
kit.check("the tables are loaded: 12 orders, 5 customers and 16 item lines",
          (loaded["orders"], loaded["customers"], loaded["item_lines"]) == (12, 5, 16), str(loaded))
kit.check("one amount is missing and two orders are cancelled, as the mini project asks",
          (loaded["missing_amounts"], loaded["cancelled"]) == (1, 2), str(loaded))

## Step 2. Redraw the pipeline (5 minutes)

Before any query, close the chapter and draw its picture on paper from memory: the six stages a
query passes through, in the order the database runs them, each with the one thing it does to the
rows. A query is written SELECT, FROM, WHERE, GROUP BY, HAVING, ORDER BY, and the picture runs in a
different order. Then run the next cell and compare the two drawings.

In [ ]:
PIPELINE = ["FROM, JOIN\nbuild the rows", "WHERE\ndrop rows", "GROUP BY\ncollapse rows",
            "HAVING\ndrop groups", "SELECT\ncompute columns", "ORDER BY\nsort output"]
kit.flow(PIPELINE, lit=1, title="The pipeline, in the order the database runs a query, with WHERE marked as in the chapter's figure")

If your drawing started with SELECT, it followed the written order, and each trap in Step 5 comes
from reading a query that way. An aggregate such as `SUM` exists only from GROUP BY onward, which is
why WHERE cannot use one and HAVING can.

## Step 3. Predict, then run the four queries (35 minutes)

Each of the four queries becomes one entry in `answers.sql`. Before each one runs, a PREDICT TODO asks
what it will return, and a check compares your prediction with the result, so a wrong picture of the
pipeline shows up as a red line rather than a shrug.

### 3.1 Revenue and order count by tier and quarter, for paid orders (10 minutes)

This is Meera's question in one statement: the paid revenue of each tier in each quarter, with the
number of orders behind it. Basic has two paid orders in Q2, and one of them carries no amount, which
is the row the diagnostic's Q13 turned on.

In [ ]:
# PREDICT, TODO 1. What will the query show for Basic in Q2, as (orders, revenue)?
#   a) (2, 500)
#   b) (1, 500)
#   c) (2, None)
#   d) (2, 250)
basic_q2_prediction = __TODO1__
print("Your prediction for Basic in Q2:", basic_q2_prediction)

In [ ]:
# TODO 2. Which pair counts every paid order and adds up the amounts that are known?
#   a) "COUNT(amount) AS orders, SUM(amount) AS revenue"
#   b) "COUNT(*) AS orders, COUNT(amount) AS revenue"
#   c) "COUNT(*) AS orders, SUM(amount) AS revenue"
#   d) "SUM(1) AS orders, AVG(amount) AS revenue"
aggregates = __TODO2__

query_1 = f"""
SELECT tier, quarter, {aggregates}
FROM orders
WHERE status = 'paid'
GROUP BY tier, quarter
ORDER BY tier, quarter
"""
result_1 = kit.sql(query_1)
kit.sql_table(query_1, caption="Query 1: paid revenue and orders by tier and quarter")

In [ ]:
basic_q2 = next(r for r in result_1 if r["tier"] == "Basic" and r["quarter"] == "Q2")
plus = {r["quarter"]: r["revenue"] for r in result_1 if r["tier"] == "Plus"}
kit.check("your prediction for Basic in Q2 matches the query",
          basic_q2_prediction == (basic_q2["orders"], basic_q2["revenue"]),
          f"you said {basic_q2_prediction}; the query says ({basic_q2['orders']}, {basic_q2['revenue']})")
kit.check("six rows, one for each tier in each quarter", len(result_1) == 6, f"{len(result_1)} rows")
kit.check("Plus earned Rs 3,749 in Q1 and Rs 2,549 in Q2", plus == {"Q1": 3749, "Q2": 2549}, str(plus))

tiers = ["Plus", "Basic", "Student"]
revenue = {(r["tier"], r["quarter"]): r["revenue"] or 0 for r in result_1}
kit.columns(tiers, [("Q1", [revenue[(t, "Q1")] for t in tiers]), ("Q2", [revenue[(t, "Q2")] for t in tiers])],
            title="Paid revenue by tier, Rs: Plus carries most of the fall", lit=(0,))

### 3.2 The average order value by tier, as a decimal (8 minutes)

The second query divides revenue by orders inside each tier, over both quarters. Two things decide its
answer: which rows the division counts, and whether the division keeps its decimals.

In [ ]:
# PREDICT, TODO 3. Basic has three paid orders, one of them with no amount, and its paid amounts
# add up to Rs 1,225. What does AVG(amount) divide that Rs 1,225 by?
#   a) 3
#   b) 4
#   c) 1
#   d) 2
basic_divisor_prediction = __TODO3__
print("Your prediction of the divisor:", basic_divisor_prediction)

In [ ]:
# TODO 4. Which expression gives each tier's average order value as a decimal?
#   a) "SUM(amount) / COUNT(amount)"
#   b) "ROUND(AVG(amount), 2)"
#   c) "ROUND(SUM(amount) / COUNT(amount), 2)"
#   d) "ROUND(SUM(amount) / COUNT(*), 2)"
average = __TODO4__

query_2 = f"""
SELECT tier, {average} AS avg_order_value, COUNT(*) AS paid_orders, COUNT(amount) AS with_amount
FROM orders
WHERE status = 'paid'
GROUP BY tier
ORDER BY tier
"""
result_2 = kit.sql(query_2)
kit.sql_table(query_2, caption="Query 2: the average order value by tier, over paid orders")

In [ ]:
basic = next(r for r in result_2 if r["tier"] == "Basic")
kit.check("your prediction of Basic's divisor matches what AVG used",
          basic_divisor_prediction == basic["with_amount"],
          f"you said {basic_divisor_prediction}; AVG divided by {basic['with_amount']}")
kit.check("Basic averages Rs 612.50, which is Rs 1,225 over the two known amounts",
          float(basic["avg_order_value"]) == 612.5, str(basic["avg_order_value"]))
kit.check("Plus averages Rs 1,259.60 across its five paid orders",
          any(r["tier"] == "Plus" and float(r["avg_order_value"]) == 1259.6 for r in result_2))

### 3.3 Every customer with their count of Q2 orders, including zero (9 minutes)

The third query starts from `customers` and joins each customer's Q2 orders, so a customer who never
ordered still gets a row. The date test is a half-open interval, on or after the first day of Q2 and
before the first day of Q3, and it sits in the `ON` clause, which filters the orders side before the
join. Cancelled orders count here, since the question is about orders placed.

In [ ]:
# PREDICT, TODO 5. How many rows will the query return?
#   a) 5
#   b) 4
#   c) 6
#   d) 12
q3_rows_prediction = __TODO5__
print("Your prediction of the rows:", q3_rows_prediction)

In [ ]:
# TODO 6. Which count gives a customer with no Q2 orders a zero?
#   a) "COUNT(*)"
#   b) "COUNT(c.customer_id)"
#   c) "COUNT(o.order_id)"
#   d) "SUM(o.amount)"
q2_count = __TODO6__

query_3 = f"""
SELECT c.customer_id, c.city, {q2_count} AS q2_orders
FROM customers c
LEFT JOIN orders o
       ON o.customer_id = c.customer_id
      AND o.order_date >= '2026-04-01' AND o.order_date < '2026-07-01'
GROUP BY c.customer_id, c.city
ORDER BY c.customer_id
"""
result_3 = kit.sql(query_3)
kit.sql_table(query_3, caption="Query 3: every customer with their Q2 orders, cancelled ones included")

In [ ]:
q3 = {r["customer_id"]: r["q2_orders"] for r in result_3}
kit.check("your prediction of the row count matches", q3_rows_prediction == len(result_3),
          f"you said {q3_rows_prediction}; the query returned {len(result_3)}")
kit.check("C5, who never ordered, appears with 0", q3.get("C5") == 0, f"C5: {q3.get('C5')}")
kit.check("the counts add up to the six orders placed in Q2", sum(v or 0 for v in q3.values()) == 6, str(q3))

### 3.4 The latest order per customer, with ROW_NUMBER (8 minutes)

`GROUP BY customer_id` would collapse each customer's orders into one row and lose which order was the
latest. A window function keeps every row and numbers each customer's orders, newest first, and an
outer query keeps the rows numbered 1. The filter sits in the outer query because the numbers do not
exist yet when the inner query's WHERE runs.

In [ ]:
# PREDICT, TODO 7. How many rows will the query return?
#   a) 12
#   b) 5
#   c) 1
#   d) 4
q4_rows_prediction = __TODO7__
print("Your prediction of the rows:", q4_rows_prediction)

In [ ]:
# TODO 8. Which window numbers each customer's orders from the newest down?
#   a) "PARTITION BY customer_id ORDER BY order_date DESC"
#   b) "PARTITION BY customer_id ORDER BY order_date"
#   c) "PARTITION BY order_date ORDER BY customer_id DESC"
#   d) "ORDER BY order_date DESC"
window = __TODO8__

query_4 = f"""
SELECT customer_id, order_id, order_date, amount, status
FROM (
    SELECT o.*, ROW_NUMBER() OVER ({window}) AS rn
    FROM orders o
) numbered
WHERE rn = 1
ORDER BY customer_id
"""
result_4 = kit.sql(query_4)
kit.sql_table(query_4, caption="Query 4: each customer's latest order, whatever its status")

In [ ]:
latest = {r["customer_id"]: r["order_id"] for r in result_4}
kit.check("your prediction of the row count matches", q4_rows_prediction == len(result_4),
          f"you said {q4_rows_prediction}; the query returned {len(result_4)}")
kit.check("each customer's newest order: C1 order 9, C2 order 11, C3 order 10, C4 order 12",
          latest == {"C1": 9, "C2": 11, "C3": 10, "C4": 12}, str(latest))

C1's latest order, number 9, was cancelled. The query answers "the latest order placed"; a
`WHERE status = 'paid'` inside the inner query would answer "the latest order paid", which is a
different question to put to Meera.

## Step 4. Trace one query through the pipeline (15 minutes)

The query in the next cell is the chapter's own Q2 revenue-by-tier query with one line added,
`HAVING COUNT(*) > 1`, so that every stage of the pipeline has work to do. Before running anything
below it, follow the twelve orders from Step 1 through the stages by hand and write down what leaves
each one.

In [ ]:
trace_query = """
SELECT tier, COUNT(*) AS orders, SUM(amount) AS revenue
FROM orders
WHERE status = 'paid' AND order_date >= '2026-04-01' AND order_date < '2026-07-01'
GROUP BY tier
HAVING COUNT(*) > 1
ORDER BY revenue DESC
"""

# TODO 9 to 13. Your trace, filled in by hand: what leaves each stage.
my_trace = [
    ("FROM orders", 12),                           # rows in
    ("WHERE paid, in Q2", __TODO9__),              # rows left
    ("GROUP BY tier", __TODO10__),                 # groups formed
    ("HAVING COUNT(*) > 1", __TODO11__),           # groups kept
    ("SELECT tier, orders, revenue", __TODO12__),  # rows in the result
    ("ORDER BY revenue DESC", __TODO13__),         # the tier on the first row, as text
]
for stage, value in my_trace:
    print(f"{stage:30} {value}")

In [ ]:
where = "status = 'paid' AND order_date >= '2026-04-01' AND order_date < '2026-07-01'"
result_t = kit.sql(trace_query)
real_trace = [
    ("FROM orders", kit.sql("SELECT COUNT(*) AS n FROM orders")[0]["n"]),
    ("WHERE paid, in Q2", kit.sql(f"SELECT COUNT(*) AS n FROM orders WHERE {where}")[0]["n"]),
    ("GROUP BY tier", kit.sql(f"SELECT COUNT(*) AS n FROM (SELECT tier FROM orders WHERE {where} "
                              f"GROUP BY tier) g")[0]["n"]),
    ("HAVING COUNT(*) > 1", kit.sql(f"SELECT COUNT(*) AS n FROM (SELECT tier FROM orders WHERE {where} "
                                    f"GROUP BY tier HAVING COUNT(*) > 1) g")[0]["n"]),
    ("SELECT tier, orders, revenue", len(result_t)),
    ("ORDER BY revenue DESC", result_t[0]["tier"]),
]
kit.table(["stage", "your trace", "the database", "same"],
          [(stage, mine, real, "yes" if mine == real else "no")
           for (stage, mine), (_, real) in zip(my_trace, real_trace)],
          caption="Twelve orders through the pipeline, stage by stage")
kit.flow([f"{stage}\n{real}" for stage, real in real_trace], lit=3,
         title="What leaves each stage, counted by the database")

In [ ]:
kit.check("your trace matches the database at every stage", my_trace == real_trace,
          f"{sum(m == r for m, r in zip(my_trace, real_trace))} of 6 stages match")
kit.check("HAVING dropped exactly one group, Student, which had a single paid order in Q2",
          real_trace[2][1] - real_trace[3][1] == 1 and "Student" not in [r["tier"] for r in result_t])

## Step 5. Break it on purpose (25 minutes)

Three changes a hurried analyst makes, each to a query you have already written correctly, and each
returning a plausible table with no error. For each one: predict what the change does, run it, read
the number that is wrong, and read the check that catches it before the fix goes back in.

### 5.1 The date test moved into WHERE (8 minutes)

The tempting change: the Q2 test looks like a filter, so it moves out of `ON` into a `WHERE` after the
join, and the query still says `LEFT JOIN`. The diagnostic's Q17 was this change, and it is the
second interview question at the end of Chapter 2.

In [ ]:
# PREDICT, TODO 14. How many rows will the query with the date test in WHERE return?
#   a) 5
#   b) 4
#   c) 6
#   d) 0
where_rows_prediction = __TODO14__

broken_3 = f"""
SELECT c.customer_id, c.city, {q2_count} AS q2_orders
FROM customers c
LEFT JOIN orders o
       ON o.customer_id = c.customer_id
WHERE o.order_date >= '2026-04-01' AND o.order_date < '2026-07-01'
GROUP BY c.customer_id, c.city
ORDER BY c.customer_id
"""
broken_3_rows = kit.sql(broken_3)
kit.sql_table(broken_3, caption="The date test in WHERE: one customer is missing")

**The plausible wrong answer.** Four customers, each with a sensible count, and no error. **Why it
is wrong.** For C5 the join produced one row with NULL in every order column; WHERE compared that NULL
date with the first of April, the comparison came out unknown, and WHERE keeps only rows that are
true. Sent to Finance as "every customer with their Q2 orders", this table says every customer
ordered in Q2, and nobody follows up the customer who has never bought anything.

In [ ]:
broken_ids = [r["customer_id"] for r in broken_3_rows]
kit.check("your prediction for the WHERE version matches", where_rows_prediction == len(broken_3_rows),
          f"you said {where_rows_prediction}; it returned {len(broken_3_rows)}")
kit.check("the check that catches it: C5 is missing, so the rows fall short of the five customers",
          "C5" not in broken_ids and len(broken_3_rows) < 5, f"customers returned: {broken_ids}")

**The fix.** Move the date test back into `ON`, where it filters the orders before the join and
every customer survives. That is query 3 as you wrote it.

In [ ]:
fixed_ids = [r["customer_id"] for r in kit.sql(query_3)]
kit.check("with the date test in ON, all five customers are back", fixed_ids == ["C1", "C2", "C3", "C4", "C5"],
          str(fixed_ids))

### 5.2 The average written as a division of two counts (7 minutes)

The tempting change: an average is a total over a count, so `SUM(amount) / COUNT(amount)` reads like
the definition. Both sides are whole numbers, and PostgreSQL divides whole numbers as whole numbers.
This is the diagnostic's Q14, and the chapter's dashboard that lies.

In [ ]:
# PREDICT, TODO 15. Plus's true average is Rs 1,259.60. What will the division show for Plus?
#   a) 1259.6
#   b) 1260
#   c) 1259
#   d) 0
plus_division_prediction = __TODO15__

broken_2 = """
SELECT tier, SUM(amount) / COUNT(amount) AS avg_order_value
FROM orders
WHERE status = 'paid'
GROUP BY tier
ORDER BY tier
"""
broken_avg = {r["tier"]: r["avg_order_value"] for r in kit.sql(broken_2)}
kit.sql_table(broken_2, caption="The average as a division of two whole numbers")

**The plausible wrong answer.** Rs 1,259, Rs 612 and Rs 362 look like averages, and each is under
a rupee short, which is why nobody notices. The same division is the one that makes a rate read zero,
and a rate is where it costs a decision. The next cell writes the cancel rate by tier both ways.

In [ ]:
cancel_rates = kit.sql("""
SELECT tier,
       SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*) AS whole_number_division,
       ROUND(SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 2) AS with_decimals
FROM orders
GROUP BY tier
ORDER BY tier
""")
kit.table(["tier", "cancel rate, whole-number division", "cancel rate, with * 1.0"],
          [(r["tier"], r["whole_number_division"], r["with_decimals"]) for r in cancel_rates],
          caption="One division, two answers")

**Why it is wrong.** A cancel-rate column that reads 0 in every tier tells Meera there are no
cancellations to chase, while a quarter of Basic's orders and a sixth of Plus's were cancelled, one of
them among the three orders Plus placed in Q2.

In [ ]:
kit.check("your prediction for Plus matches the division", plus_division_prediction == broken_avg["Plus"],
          f"you said {plus_division_prediction}; the division gave {broken_avg['Plus']}")
kit.check("the check that catches it: the division returns whole numbers, which the self-check rejects",
          all(v == int(v) for v in broken_avg.values()), str(broken_avg))
kit.check("the whole-number cancel rate reads 0 in every tier, while Basic's true rate is 0.25",
          all(r["whole_number_division"] == 0 for r in cancel_rates)
          and any(float(r["with_decimals"]) == 0.25 for r in cancel_rates))

**The fix.** Make one side of the division a decimal before dividing: `AVG(amount)`, or
`SUM(amount) * 1.0 / COUNT(amount)`, as query 2 already does.

In [ ]:
fixed_avg = {r["tier"]: float(r["avg_order_value"]) for r in kit.sql(query_2)}
kit.check("query 2 keeps the decimals the division dropped",
          all(fixed_avg[t] > broken_avg[t] for t in fixed_avg), str(fixed_avg))

### 5.3 A join that repeats rows (10 minutes)

The tempting change: suppose Kavya asks for the number of items sold beside the revenue, so query 1
gains a join to `order_items`, which holds one row for every item line on an order. Most orders have one
line; orders 7 and 9 have two and order 12 has three. Chapter 2's cheap check is the row count before
and after the join, and the self-check's first line compares the total with the raw table's.

In [ ]:
# PREDICT, TODO 16. After the join, the revenue column's total, set against the raw table's paid
# total, will be:
#   a) "lower"
#   b) "the same"
#   c) "NULL"
#   d) "higher"
joined_total_prediction = __TODO16__

fanout = """
SELECT o.tier, o.quarter, COUNT(i.line_no) AS items, SUM(o.amount) AS revenue
FROM orders o
JOIN order_items i ON i.order_id = o.order_id
WHERE o.status = 'paid'
GROUP BY o.tier, o.quarter
ORDER BY o.tier, o.quarter
"""
fanout_rows = kit.sql(fanout)
kit.sql_table(fanout, caption="Query 1 with the item lines joined in")

In [ ]:
rows_before = kit.sql("SELECT COUNT(*) AS n FROM orders WHERE status = 'paid'")[0]["n"]
rows_after = kit.sql("SELECT COUNT(*) AS n FROM orders o JOIN order_items i ON i.order_id = o.order_id "
                     "WHERE o.status = 'paid'")[0]["n"]
raw_paid = kit.sql("SELECT SUM(amount) AS total FROM orders WHERE status = 'paid'")[0]["total"]
joined_total = sum(r["revenue"] or 0 for r in fanout_rows)
joined_by_quarter = {q: sum(r["revenue"] or 0 for r in fanout_rows if r["quarter"] == q) for q in ("Q1", "Q2")}
print(f"Paid orders before the join: {rows_before}. Rows after it: {rows_after}.")
print(f"Raw paid total Rs {raw_paid:,}; the joined query's total Rs {joined_total:,}.")
print(f"The joined query says paid revenue went from Rs {joined_by_quarter['Q1']:,} in Q1 "
      f"to Rs {joined_by_quarter['Q2']:,} in Q2.")
true_by_quarter = {r["quarter"]: r["revenue"] for r in kit.sql(
    "SELECT quarter, SUM(amount) AS revenue FROM orders WHERE status = 'paid' GROUP BY quarter")}
kit.columns(["Q1", "Q2"], [("the orders table", [true_by_quarter["Q1"], true_by_quarter["Q2"]]),
                           ("after the join", [joined_by_quarter["Q1"], joined_by_quarter["Q2"]])],
            title="Paid revenue by quarter, Rs: the join turns Meera's fall into a rise", lit=(1,), width=480)

**The plausible wrong answer.** Plus reads Rs 3,749 in Q1 and Rs 6,447 in Q2, and the joined table
says paid revenue rose from Rs 4,899 to Rs 7,247. **Why it is wrong.** Every order row was repeated
once for each of its item lines before `SUM` ran, so order 12's Rs 1,349 was added three times. Taken
to Meera, this table answers her question the wrong way round: revenue did not fall, and Plus is the
tier to celebrate.

In [ ]:
direction = "higher" if joined_total > raw_paid else "lower" if joined_total < raw_paid else "the same"
kit.check("your prediction of the joined total matches", joined_total_prediction == direction,
          f"you said {joined_total_prediction!r}; the joined total is {direction}")
kit.check("the tell the chapter names: 10 paid orders became 13 rows after a join meant to add a column",
          (rows_before, rows_after) == (10, 13), f"{rows_before} rows before, {rows_after} after")
kit.check("the check that catches it: the tier totals no longer add up to the raw paid total",
          joined_total != raw_paid, f"Rs {joined_total:,} against Rs {raw_paid:,}")

**The fix.** Aggregate the many side to one row per order first, then join the totals.
`SUM(DISTINCT o.amount)` is no repair: C1's two Q1 orders both cost Rs 1,200, and DISTINCT merges them
into one.

In [ ]:
fixed_fanout = """
WITH items_per_order AS (
    SELECT order_id, COUNT(*) AS items
    FROM order_items
    GROUP BY order_id
)
SELECT o.tier, o.quarter, SUM(i.items) AS items, SUM(o.amount) AS revenue
FROM orders o
JOIN items_per_order i ON i.order_id = o.order_id
WHERE o.status = 'paid'
GROUP BY o.tier, o.quarter
ORDER BY o.tier, o.quarter
"""
fixed_rows = kit.sql(fixed_fanout)
kit.sql_table(fixed_fanout, caption="Items counted per order first, then joined")
distinct_plus_q1 = kit.sql("""
SELECT SUM(DISTINCT o.amount) AS revenue
FROM orders o
JOIN order_items i ON i.order_id = o.order_id
WHERE o.status = 'paid' AND o.tier = 'Plus' AND o.quarter = 'Q1'
""")[0]["revenue"]
kit.check("with the items counted per order first, revenue adds up to the raw paid total again",
          sum(r["revenue"] or 0 for r in fixed_rows) == raw_paid and sum(r["items"] for r in fixed_rows) == 13)
kit.check("SUM(DISTINCT) gives Plus Rs 2,549 in Q1 against the true Rs 3,749, so it repairs nothing",
          distinct_plus_q1 == 2549, f"Rs {distinct_plus_q1:,}")

## Step 6. The mini project's self-check (5 minutes)

The guide's four self-check lines, run against your four queries.

In [ ]:
tier_total = sum(r["revenue"] or 0 for r in result_1)
customers_with_orders = kit.sql("SELECT COUNT(DISTINCT customer_id) AS n FROM orders")[0]["n"]
q3_now = {r["customer_id"]: r["q2_orders"] for r in kit.sql(query_3)}
q4_now = kit.sql(query_4)
kit.check("the tier totals add up to the raw table's paid total", tier_total == raw_paid,
          f"Rs {tier_total:,} against Rs {raw_paid:,}")
kit.check("the customer with no orders appears with 0", q3_now.get("C5") == 0, str(q3_now))
kit.check("the average order value is not a whole number",
          all(r["avg_order_value"] != int(r["avg_order_value"]) for r in result_2),
          str({r["tier"]: str(r["avg_order_value"]) for r in result_2}))
kit.check("the ROW_NUMBER query returns exactly one row per customer who has ordered",
          len(q4_now) == customers_with_orders == len({r["customer_id"] for r in q4_now}),
          f"{len(q4_now)} rows for {customers_with_orders} customers with orders")

The next cell prints what goes into the repository: the four queries as `answers.sql` holds them,
then their four result tables as markdown, ready to paste into the README.

In [ ]:
def as_markdown(rows):
    heads = list(rows[0])
    lines = ["| " + " | ".join(heads) + " |", "|" + " --- |" * len(heads)]
    lines += ["| " + " | ".join("" if r[h] is None else str(r[h]) for h in heads) + " |" for r in rows]
    return "\n".join(lines)


final_queries = [query_1, query_2, query_3, query_4]
print("-- answers.sql")
for n, q in enumerate(final_queries, start=1):
    print(f"\n-- Query {n}\n{q.strip()};")
for n, q in enumerate(final_queries, start=1):
    print(f"\nQuery {n}\n\n{as_markdown(kit.sql(q))}")

In [ ]:
kit.check_summary()

## What you would now say to Meera

One sentence, with every number in it taken from a query you checked: how far paid revenue fell, how
much of the fall is Plus, what happened to Plus's orders, and what the order with no amount does to
Basic's figure. The worked notebook has one version to set yours against.

## Into your `w00-diagnostic-sql` repository

Chapter 8 shows how to create the repository and the five-heading README every repository carries.
Three files go into it from here:

- `orders.sql`: the setup file, copied as it is, or your own twelve rows in the same shape. The
  `order_items` table serves Step 5 only and can be left out.
- `answers.sql`: the four queries exactly as the cell before the summary prints them, in order.
- `README.md`: the four result tables that cell prints, pasted under the Result heading, with your
  sentence to Meera above them.

Run `answers.sql` once against your `orders.sql` before you commit, and read its four tables against
the self-check above. Then post the repository's link in the Week 0 thread on Discussions, as the
guide's path asks.